# Can we predict 12-month loan default?
**Target:** `defaulted` (1 = defaulted within 12 months after funding). **Data:** `data.csv` (one row per application), described in `columns.csv`.

Plan: data quality, ID/leakage check, baseline, logistic regression and gradient boosting compared by cross-validation on a development set, one-time evaluation on a held-out test set, interpretation.
Metrics: ROC AUC and PR AUC (about 10% positives), plus lift by score decile.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
SEED = 42
df = pd.read_csv('data.csv'); dd = pd.read_csv('columns.csv')
dd

## 1. Data quality

In [ ]:
print(df.shape)
print('missing values:\n', df.isna().sum().to_string())
print('duplicate rows:', df.duplicated().sum(), '| duplicate ids:', df.application_id.duplicated().sum())
print('default rate:', df.defaulted.mean().round(4))
df.describe().T

In [ ]:
num = ['income_k', 'debt_ratio', 'credit_lines', 'age', 'employment_years', 'loan_amount_k']
print((df[['income_k', 'loan_amount_k', 'employment_years', 'age', 'credit_lines']] < 0).sum().to_string())
print('employment_years > age-16:', (df.employment_years > df.age - 16).sum())
print('employment_years == 0:', (df.employment_years == 0).mean().round(3))
print('debt_ratio > 1:', (df.debt_ratio > 1).sum())
print(df.region.value_counts().to_string())

## 2. Univariate relationship with default

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(14, 7))
for a, c in zip(ax.ravel(), num):
    q = pd.qcut(df[c], 10, duplicates='drop')
    g = df.groupby(q, observed=True).defaulted.mean()
    a.plot(range(len(g)), g.values, 'o-'); a.axhline(df.defaulted.mean(), ls='--', c='grey')
    a.set_title(c + ' (decile)'); a.set_ylabel('default rate'); a.set_xlabel('decile (low to high)')
plt.tight_layout(); plt.show()
print(df.groupby('region').defaulted.agg(['mean', 'size']))

In [ ]:
# application_id should carry no signal; check it is not a leak (e.g. ordering)
print('AUC of application_id alone:', round(roc_auc_score(df.defaulted, df.application_id), 3))
print(df[num + ['defaulted']].corr()['defaulted'].round(3).to_string())

**Observations:** no missing values or duplicates. Default rate rises with `debt_ratio` and `loan_amount_k`, and falls with `income_k` and `employment_years`. `age`, `credit_lines` and `region` show little marginal signal. `application_id` is uninformative (AUC about 0.50), so there is no ordering leak. Data oddity: 643 rows have `employment_years` greater than `age - 16`, i.e. implausibly long tenure for the age; worth checking at the source (rows were kept, since the tree and linear models are not sensitive to them).

## 3. Split
There is no date column, so a time-based split is not possible; I use a stratified random split. 20% is held out and used only once at the end. `application_id` is dropped (identifier).

In [ ]:
X = df.drop(columns=['defaulted', 'application_id']); y = df.defaulted
Xdev, Xte, ydev, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print(Xdev.shape, Xte.shape, ydev.mean().round(4), yte.mean().round(4))

## 4. Models (5-fold CV on the development set)

In [ ]:
pre = ColumnTransformer([('num', StandardScaler(), num), ('reg', OneHotEncoder(drop='first'), ['region'])])
models = {
    'baseline (prior)': DummyClassifier(strategy='prior'),
    'logistic regression': make_pipeline(pre, LogisticRegression(max_iter=1000)),
    'gradient boosting': make_pipeline(
        ColumnTransformer([('n', 'passthrough', num), ('r', OneHotEncoder(), ['region'])]),
        HistGradientBoostingClassifier(learning_rate=0.05, max_iter=200, max_depth=3,
                                       min_samples_leaf=50, l2_regularization=1.0, random_state=SEED)),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
oof = {}; rows = []
for n, m in models.items():
    p = cross_val_predict(m, Xdev, ydev, cv=cv, method='predict_proba')[:, 1]; oof[n] = p
    rows.append(dict(model=n, roc_auc=roc_auc_score(ydev, p), pr_auc=average_precision_score(ydev, p),
                     brier=brier_score_loss(ydev, p)))
pd.DataFrame(rows).set_index('model').round(4)

In [ ]:
for n in ['logistic regression', 'gradient boosting']:
    a = [roc_auc_score(ydev.iloc[te], oof[n][te]) for _, te in cv.split(Xdev, ydev)]
    print(f'{n}: AUC per fold {np.round(a, 3)}  mean {np.mean(a):.3f} +/- {np.std(a):.3f}')

## 5. Final evaluation on the held-out test set

In [ ]:
res = {}; fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for n, m in models.items():
    m.fit(Xdev, ydev); p = m.predict_proba(Xte)[:, 1]; res[n] = p
    if n.startswith('baseline'): continue
    f, t, _ = roc_curve(yte, p); ax[0].plot(f, t, label=f'{n} (AUC {roc_auc_score(yte, p):.3f})')
    pr, rc, _ = precision_recall_curve(yte, p); ax[1].plot(rc, pr, label=f'{n} (AP {average_precision_score(yte, p):.3f})')
ax[0].plot([0, 1], [0, 1], 'k--'); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR'); ax[0].set_title('ROC')
ax[1].axhline(yte.mean(), c='k', ls='--'); ax[1].set_xlabel('recall'); ax[1].set_ylabel('precision'); ax[1].set_title('Precision-recall')
for a in ax: a.legend()
plt.show()
rng = np.random.default_rng(SEED); yt = yte.values
for n in ['logistic regression', 'gradient boosting']:
    b = []
    for _ in range(1000):
        i = rng.integers(0, len(yt), len(yt)); b.append(roc_auc_score(yt[i], res[n][i]))
    print(f'{n}: test AUC {roc_auc_score(yt, res[n]):.3f}  95% bootstrap CI [{np.percentile(b, 2.5):.3f}, {np.percentile(b, 97.5):.3f}]')
print('test AP, base rate:', round(yte.mean(), 3), {n: round(average_precision_score(yte, res[n]), 3) for n in res})

In [ ]:
p = res['logistic regression']; t = pd.DataFrame({'p': p, 'y': yte.values})
t['decile'] = pd.qcut(t.p.rank(method='first'), 10, labels=range(1, 11))
g = t.groupby('decile', observed=True).agg(applicants=('y', 'size'), defaults=('y', 'sum'), rate=('y', 'mean'), mean_pred=('p', 'mean'))
g['lift'] = g.rate / yte.mean(); print(g.round(3).iloc[::-1])
print('Share of all defaults captured in the top 20% by score:', round(t.sort_values('p', ascending=False).head(len(t) // 5).y.sum() / t.y.sum(), 3))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for n in ['logistic regression', 'gradient boosting']:
    fr, mp = calibration_curve(yte, res[n], n_bins=8, strategy='quantile'); ax.plot(mp, fr, 'o-', label=n)
ax.plot([0, .5], [0, .5], 'k--'); ax.set_xlabel('predicted'); ax.set_ylabel('observed'); ax.set_title('Calibration (test)'); ax.legend(); plt.show()

**Reading the test results:** the models rank risk well above chance but are far from precise. Default rate rises from about 1% in the lowest-scored decile to about 36% in the highest (3.5x the base rate), and the top 20% of scores contains about half of all defaults. Predicted probabilities track observed rates reasonably well. Gradient boosting does not beat the simpler logistic regression, which suggests the signal is mostly additive and roughly linear.

## 6. What drives the predictions?

In [ ]:
lr = models['logistic regression']
print(pd.Series(lr[-1].coef_[0], index=lr[0].get_feature_names_out()).sort_values().round(3).to_string())
pi = permutation_importance(models['gradient boosting'], Xte, yte, scoring='roc_auc', n_repeats=20, random_state=SEED)
s = pd.Series(pi.importances_mean, index=Xte.columns); e = pd.Series(pi.importances_std, index=Xte.columns)
o = s.sort_values().index; s[o].plot.barh(xerr=e[o])
plt.xlabel('drop in test AUC when shuffled'); plt.title('Permutation importance (gradient boosting)'); plt.show()

## 7. Conclusion
**Yes, default is predictable to a useful but limited degree.** Held-out ROC AUC is about 0.77 (95% CI roughly 0.74 to 0.81) against 0.50 for a no-skill baseline, and PR AUC is about 0.35 against a 10% base rate.

* Main drivers: higher `debt_ratio` and `loan_amount_k` raise risk; higher `income_k` and longer `employment_years` lower it. `age`, `credit_lines` and `region` add little.
* A simple, interpretable logistic regression is as good as gradient boosting; prefer it.
* Practical use: as a risk-ranking / triage score it is helpful (top decile about 3.5x base rate). It is not accurate enough to make individual accept/reject calls on its own: even in the top decile roughly 2 in 3 loans do not default.

**Caveats:** random (not time-based) split because there is no date, so performance on future applications may be lower if the population drifts; only about 840 defaults in total, hence the wide confidence intervals; the data holds funded loans only, so the model says nothing about applicants who were declined; the relationship is correlational, and the `employment_years` anomaly noted above should be checked.